# UK Gas Demand Forecast — Exploration & EDA

This notebook walks through:
1. Loading demand and weather data from the public APIs
2. Exploratory analysis of demand seasonality and weather correlations
3. Feature engineering validation
4. Initial XGBoost model training and evaluation
5. Walk-forward backtest results

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.data.demand_client import DemandClient
from src.data.weather_client import WeatherClient
from src.data import cache
from src.features.engineer import build_features, FEATURE_COLS
from src.model.xgboost_model import GasDemandXGB
from src.model.backtest import walk_forward_xgb
from src.model.evaluate import compute_metrics, residual_analysis

pd.set_option('display.max_columns', 50)

## 1. Load Data

In [ ]:
# Demand data from National Gas API
demand = cache.load("nts_demand")
if demand is None:
    client = DemandClient()
    demand = client.get_all_demand(start="2020-10-01")
    if demand is not None:
        cache.save("nts_demand", demand)

print(f"Demand: {len(demand)} rows, {demand['date'].min()} to {demand['date'].max()}")
demand.head()

In [ ]:
# Weather data from Open-Meteo API
weather = cache.load("weather_historical")
if weather is None:
    wc = WeatherClient()
    weather = wc.get_historical(start="2020-10-01")
    if weather is not None:
        cache.save("weather_historical", weather)

print(f"Weather: {len(weather)} rows, {weather['date'].min()} to {weather['date'].max()}")
weather.head()

## 2. Demand Seasonality

In [ ]:
demand_plot = demand.copy()
demand_plot['date'] = pd.to_datetime(demand_plot['date'])

fig = px.line(demand_plot, x='date', y='demand_mcm',
              title='UK NTS Daily Gas Demand (mcm/d)',
              labels={'demand_mcm': 'Demand (mcm/d)'})
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [ ]:
demand_plot['month'] = demand_plot['date'].dt.month
monthly = demand_plot.groupby('month')['demand_mcm'].agg(['mean', 'std']).reset_index()

fig = px.bar(monthly, x='month', y='mean', error_y='std',
             title='Monthly Average Demand',
             labels={'mean': 'Mean Demand (mcm/d)'})
fig.update_layout(template='plotly_dark', height=350)
fig.show()

## 3. Weather Correlations

In [ ]:
df = build_features(demand, weather)
print(f"Feature matrix: {df.shape}")
df[['date', 'demand_mcm', 'temp_mean', 'hdd', 'windspeed_max']].describe()

In [ ]:
fig = px.scatter(df, x='temp_mean', y='demand_mcm', color='is_winter',
                 title='Temperature vs Demand',
                 trendline='ols', opacity=0.4)
fig.update_layout(template='plotly_dark', height=400)
fig.update_traces(marker_size=3)
fig.show()

In [ ]:
fig = px.scatter(df, x='hdd', y='demand_mcm', color='gas_quarter',
                 title='HDD vs Demand', trendline='ols', opacity=0.4)
fig.update_layout(template='plotly_dark', height=400)
fig.update_traces(marker_size=3)
fig.show()

In [ ]:
# Correlation matrix of key features
corr_cols = ['demand_mcm', 'temp_mean', 'hdd', 'effective_hdd', 'windspeed_max',
             'demand_lag_1', 'demand_lag_7', 'is_weekend']
corr = df[corr_cols].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='Feature Correlation Matrix')
fig.update_layout(template='plotly_dark', height=500)
fig.show()

## 4. XGBoost Model Training

In [ ]:
model = GasDemandXGB()
model.fit(df)

preds = model.predict(df)
in_sample = compute_metrics(df['demand_mcm'].values, preds)
print("In-sample metrics:")
for k, v in in_sample.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
fi = model.feature_importance()
fig = px.bar(fi.head(15), x='importance', y='feature', orientation='h',
             title='Top 15 Feature Importances',
             color='importance', color_continuous_scale='Oranges')
fig.update_layout(template='plotly_dark', height=400)
fig.update_yaxes(autorange='reversed')
fig.show()

## 5. Walk-Forward Backtest

In [ ]:
bt = walk_forward_xgb(df)
print("\nBacktest metrics:")
bt.metrics_table

In [ ]:
all_preds = bt.all_predictions
all_preds['residual'] = all_preds['actual'] - all_preds['predicted']

fig = go.Figure()
fig.add_trace(go.Scatter(x=all_preds['date'], y=all_preds['actual'],
                         name='Actual', line=dict(color='#3498db', width=1)))
fig.add_trace(go.Scatter(x=all_preds['date'], y=all_preds['predicted'],
                         name='Predicted', line=dict(color='#e67e22', width=1)))
fig.update_layout(template='plotly_dark', height=400,
                  title='Walk-Forward: Actual vs Predicted (Out-of-Sample)',
                  yaxis_title='Demand (mcm/d)')
fig.show()

In [ ]:
fig = px.histogram(all_preds, x='residual', nbins=50,
                   title='Backtest Residual Distribution',
                   color_discrete_sequence=['#e67e22'])
fig.update_layout(template='plotly_dark', height=350)
fig.show()

print(f"Residual mean: {all_preds['residual'].mean():.2f} mcm/d")
print(f"Residual std:  {all_preds['residual'].std():.2f} mcm/d")